## Notebook for Ingestion testing

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

C:\Users\aadit\AppData\Local\Temp\ipykernel_21400\4179884828.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [5]:
chroma_dir = './chroma_db'
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


c:\Users\aadit\Desktop\financial-qa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2261.94it/s]


In [23]:
source_doc = "pdf_files"
doc_name = "10-k-Annual-report.pdf"

In [31]:
# Load Document
loader = PyPDFLoader(
        file_path = f"{source_doc}/{doc_name}")
# docs = loader.load()

In [52]:
txt_splittor = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " "], 
    length_function=len,   # Function used to calculate chunk length (default is len)
    add_start_index=True   # Get you the start index of the Text
)

In [57]:

for page_num, page in enumerate(loader.lazy_load()):
    # Process each page-level document individually
    page_chunks = txt_splittor.split_documents([page])

    for chunk_number,chunk in enumerate(page_chunks):
        print(f"--- Page number {page_num+1} --- chunk_number {chunk_number+1} start_index {chunk.metadata['start_index']}\n")
        # print(f"Content: {chunk.page_content}")
        print(f"Metadata: {chunk.metadata}\n")


--- Page number 1 --- chunk_number 1 start_index 0

Metadata: {'producer': 'Skia/PDF m140', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36', 'creationdate': '2026-10-09T02:47:06+00:00', 'title': 'aapl-20250927', 'moddate': '2026-10-09T02:47:06+00:00', 'source': 'pdf_files/10-k-Annual-report.pdf', 'total_pages': 78, 'page': 0, 'page_label': '1', 'start_index': 0}

--- Page number 1 --- chunk_number 2 start_index 849

Metadata: {'producer': 'Skia/PDF m140', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36', 'creationdate': '2026-10-09T02:47:06+00:00', 'title': 'aapl-20250927', 'moddate': '2026-10-09T02:47:06+00:00', 'source': 'pdf_files/10-k-Annual-report.pdf', 'total_pages': 78, 'page': 0, 'page_label': '1', 'start_index': 849}

--- Page number 2 --- chunk_number 1 start_index 0

Metadata: {'producer': 'Skia/PDF m140', 'creator': 'Mozill

In [8]:
from graph import graph

print(graph.invoke({"question": "What was Google’s total net sales in fiscal 2025?"})["answer"])

No relevant content was found in the uploaded documents.


In [1]:
from helper import vector_db

store = vector_db()
for q in [
        "What was Apple's total net sales in fiscal 2025?",
        "What was gOOGLE's total net sales in fiscal 2025?",
    ]:
    print("QUESTION:", q)
    for doc, score in store.similarity_search_with_relevance_scores(q, k=5):
        print(round(score, 3), doc.metadata.get("page", 0) + 1, doc.page_content[:60].replace("\n", " "))
    print()

c:\Users\aadit\Desktop\financial-qa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


QUESTION: What was Apple's total net sales in fiscal 2025?
0.597 32 Products and Services Performance The following table shows 
0.565 53 of total vendor non-trade receivables, which accounted for 4
0.543 41 Apple Inc. CONSOLIDATED STATEMENTS OF OPERATIONS (In million
0.534 32 by lower net sales of iPad Pro. Wearables, Home and Accessor
0.517 31 net sales of Mac. Japan Japan net sales increased during 202

QUESTION: What was gOOGLE's total net sales in fiscal 2025?
0.391 49 The following table shows disaggregated net sales, as well a
0.373 31 operations. Segment Operating Performance The following tabl
0.373 65 The following tables show net sales for 2025, 2024 and 2023 
0.35 32 Products and Services Performance The following table shows 
0.34 32 by lower net sales of iPad Pro. Wearables, Home and Accessor

